In [ ]:
import pandas as pd
import ipaddress

from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("malicious_phish.csv")

print("Original dataset:")
print(df.head())
print("\nColumns:")
print(df.columns)

Original dataset:
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement

Columns:
Index(['url', 'type'], dtype='object')


In [ ]:
df = df.rename(columns={
    "type": "label"
})
df = df.dropna(subset=["url", "label"])

In [ ]:
# Convert URLs to strings
df["url"] = df["url"].astype(str)

# Remove whitespace
df["url"] = df["url"].str.strip()

# Convert to lowercase
df["url"] = df["url"].str.lower()

df = df.drop_duplicates(subset=["url"])

df = df.reset_index(drop=True)

In [ ]:
label_mapping = {
    "benign": "legitimate",
    "phishing": "malicious",
    "defacement": "malicious",
    "malware": "malicious"
}


df["label"] = (
    df["label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["label"] = df["label"].map(label_mapping)


# Remove any rows with labels that were not recognised
df = df.dropna(subset=["label"])


# Convert labels to numerical values
df["label"] = df["label"].map({
    "legitimate": 0,
    "malicious": 1
})

In [ ]:
def extract_features(url):

    original_url = str(url)

    # Default values in case URL parsing fails
    domain = ""
    path = ""
    scheme = ""

    # Add protocol temporarily if one is missing
    parse_url = original_url

    if "://" not in parse_url:
        parse_url = "http://" + parse_url

    # Try to parse the URL
    try:
        parsed = urlparse(parse_url)

        domain = parsed.netloc.split(":")[0]
        path = parsed.path
        scheme = parsed.scheme

    except ValueError:
        # URL is malformed, so use empty
        # domain/path values instead
        domain = ""
        path = ""
        scheme = ""


    # Feature Extraction
    features = {

        # 1. URL length
        "url_length": len(original_url),

        # 2. Domain length
        "domain_length": len(domain),

        # 3. Path length
        "path_length": len(path),

        # 4. Number of digits
        "digit_count": sum(
            character.isdigit()
            for character in original_url
        ),

        # 5. Number of special characters
        "special_character_count": sum(
            not character.isalnum()
            for character in original_url
        ),

        # 6. Number of dots
        "dot_count": original_url.count("."),

        # 7. Number of subdomains
        "subdomain_count": max(
            0,
            len(domain.split(".")) - 2
        )
    }


    # 8. Is the domain an IP address?
    try:
        ipaddress.ip_address(domain)
        features["has_ip"] = 1

    except ValueError:
        features["has_ip"] = 0


    # 9. Does the URL use HTTPS?
    features["has_https"] = int(
        scheme == "https"
    )


    # 10. Suspicious keyword count
    suspicious_keywords = [
        "login",
        "signin",
        "verify",
        "verification",
        "account",
        "secure",
        "update",
        "password",
        "bank"
    ]

    features["suspicious_keyword_count"] = sum(
        keyword in original_url
        for keyword in suspicious_keywords
    )


    return features


In [ ]:
feature_data = df["url"].apply(extract_features)

features_df = pd.DataFrame(
    feature_data.tolist()
)

features_df["label"] = df["label"].values

In [ ]:
X = features_df.drop(columns=["label"])
y = features_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
print("Data processing complete!")

print("\nDataset size:")
print(f"Total URLs: {len(df)}")

print("\nClass distribution:")
print(
    df["label"]
    .map({
        0: "legitimate",
        1: "malicious"
    })
    .value_counts()
)

print("\nTraining data:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print("\nTesting data:")
print(f"X_test: {X_test.shape}")

print("\nFeatures:")
print(list(X.columns))

Data processing complete!

Dataset size:
Total URLs: 641053

Class distribution:
label
legitimate    428059
malicious     212994
Name: count, dtype: int64

Training data:
X_train: (512842, 10)
y_train: (512842,)

Testing data:
X_test: (128211, 10)

Features:
['url_length', 'domain_length', 'path_length', 'digit_count', 'special_character_count', 'dot_count', 'subdomain_count', 'has_ip', 'has_https', 'suspicious_keyword_count']


In [ ]:
# Cleaned raw URL dataset
df.to_csv("cleaned_urls.csv", index=False)

# Feature dataset for machine learning
features_df.to_csv("processed_url_features.csv", index=False)